# 사진 2장으로 스테레오 거리 측정 (젯슨 JupyterLab 판)

`stereo_colab_v4.ipynb`(최신)에서 **업로드·다운로드 셀만 파일 경로 방식으로 바꾼 것**입니다. 계산 셀(3~7)과 설정 기본값(56°, 33.3mm, 보정 0 — 2026-11-11 캘리브레이션 실측)은 v4 와 같습니다.
이 폴더의 `stereo_calib.npz`(스마트큐브 천장 C2(왼쪽)·C1(오른쪽), 33쌍)가 있으면 자동으로 **정확 측정 모드**로 계산합니다.

**사용법**: 위에서부터 셀을 순서대로 실행하세요 (`Shift + Enter`). 캘리브레이션(C1~C3)은 카메라를 다시 만졌을 때만 필요합니다.

| 단계 | 내용 |
|---|---|
| 0 | 준비 (라이브러리, 한글 폰트, 결과 폴더) |
| 1 | 설정값 입력 (카메라 간격, 화각 — 캘리브레이션 파일을 쓰면 무시됨) |
| C1~C3 | **(선택)** 체커보드 사진 폴더(`collect_pairs.py` 로 모은 `calib/`)로 캘리브레이션 → `stereo_calib.npz` 생성 |
| 2 | 왼쪽·오른쪽 사진 **경로** 지정 (기본: `사용한 파일/cube_L_C2_20261111.jpg`, `cube_R_C1_20261111.jpg` — 큐브 바닥 체커보드, 줄자 58.4cm) |
| 3~5 | 정렬 → 시차 계산 → 거리 계산 (자동) |
| 6 | 결과 보기 (마우스를 올리면 거리 표시) |
| 7 | 원하는 지점 거리 측정 |
| 8 | 결과 저장 (`결과/` 폴더) |

명령줄로 같은 계산을 하려면 이 폴더의 `stereo_pair.py` 를 쓰세요.


In [ ]:
#@title 0. 준비 (라이브러리, 한글 폰트, 결과 폴더)
import os, re, glob
import cv2, numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

# 젯슨에 설치된 나눔고딕(fonts-nanum)으로 그래프 한글 표시
FONT = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if os.path.exists(FONT):
    font_manager.fontManager.addfont(FONT)
    plt.rcParams["font.family"] = "NanumGothic"
else:
    print("나눔고딕 폰트가 없어 그래프 한글이 깨질 수 있습니다 (sudo apt-get install fonts-nanum)")
plt.rcParams["axes.unicode_minus"] = False

OUT_DIR = "결과"                 # 결과 저장 폴더 (이 노트북이 있는 폴더 기준)
os.makedirs(OUT_DIR, exist_ok=True)
CALIB_RESULT = None              # C2 셀에서 캘리브레이션하면 파일 경로가 저장됨
print("준비 완료, OpenCV", cv2.__version__, "| 결과 폴더:", os.path.abspath(OUT_DIR))


In [ ]:
#@title 1. 설정 { display-mode: "form" }
BASELINE_MM = 33.3     #@param {type:"number"}
#@markdown ↑ 두 카메라(촬영 위치) 사이 거리(mm). 캘리브레이션 파일을 쓰면 무시됨. 스마트큐브 천장 카메라 = 33.3 (2026-11-11 캘리브레이션 실측, 자 실측 32~33)
HFOV_DEG = 56.0        #@param {type:"number"}
#@markdown ↑ 카메라 가로 화각(도). 캘리브레이션 파일을 쓰면 무시됨. ESP32-CAM OV5640 "72°" 모듈 = 가로 약 56 (72°는 대각선 화각), OV2640 기본 렌즈 ≈ 66
RESIZE_WIDTH = 640     #@param {type:"integer"}
#@markdown ↑ 계산용 가로 해상도. 큰 사진은 줄여야 빠르고 잘 맞음 (0 = 원본 유지)
RECTIFY_MODE = "auto"  #@param ["auto", "y-align", "none"]
#@markdown ↑ auto: 캘리브레이션 파일이 있으면 사용, 없으면 y-align (특징점으로 세로 어긋남 보정)
BLOCK_SIZE = 5         #@param {type:"slider", min:3, max:15, step:2}
MIN_DIST_CM = 20       #@param {type:"integer"}
#@markdown ↑ 정확 측정 모드의 시차 탐색 범위 계산용: 이보다 가까운 물체는 측정하지 않음
MAX_DIST_CM = 500      #@param {type:"integer"}
#@markdown ↑ 이보다 먼 값은 표시하지 않음
DISP_OFFSET_PX = 0.0   #@param {type:"number"}
#@markdown ↑ 시차 보정값(px), **근사 모드에서만** 적용. 카메라가 살짝 안쪽/바깥쪽을 향해 생기는 오차 보정, 실측 2지점으로 계산 (모르면 0). 정확 측정 모드는 캘리브레이션이 대신함


---
## (선택) 캘리브레이션: 체커보드 사진으로 `stereo_calib.npz` 만들기

팀 폴더의 `stereo_calib.npz`(2026-11-11, 스마트큐브 천장 C2·C1, 33쌍)가 있으면 이 단계는 건너뛰고 셀 2에서 그 파일을 올리면 됩니다. **카메라를 다시 만지거나 해상도를 바꾸면 다시 해야 합니다.**

**준비**
- 체커보드 인쇄 (10×7칸 → 내부 코너 9×6), 평평한 판에 붙이기. 한 칸 크기를 자로 정확히 재기 (A4 인쇄본 실측 24.2mm — 프린터 배율에 따라 다름)
- 측정에 쓸 **같은 두 카메라, 같은 고정 상태**로 촬영. 한 쌍 = 같은 순간의 왼쪽 사진 + 오른쪽 사진. 체커보드 **전체**가 두 사진 모두에 보여야 함 (잘린 쌍은 자동 제외)
- **25쌍 이상**, 거리(높이)를 3~4단계로 바꾸고 그중 절반은 보드를 **20~40° 기울여서**. 평평한 보드만 찍으면 초점거리가 정해지지 않아 거리가 2~3% 틀림 (2026-11-11 확인)
- 스마트큐브: 대시보드 "동시 촬영(2대)"으로 찍고, PC에서 `python collect_pairs.py --since <시각> --out calib --left CUBE001-C2` 로 `calib/L_NN.jpg`, `R_NN.jpg` 를 모은 뒤 zip. 양쪽 썸네일이 모두 뜰 때까지 보드를 움직이지 말 것 (업로드가 실패하면 카메라가 다시 찍음)

**업로드 방법 (둘 중 하나)**
- **A. zip 파일**: `collect_pairs.py` 또는 PC의 `python stereo.py collect`로 만든 `calib` 폴더를 압축 (`L_00.jpg`, `R_00.jpg` …). 이름에 `left`/`right`가 들어간 파일도 인식
- **B. 사진 직접 선택**: 왼쪽 사진들을 한꺼번에 선택 → 이어서 오른쪽 사진들을 한꺼번에 선택. **파일 이름 순서로 짝을 지으므로** 촬영 순서가 같아야 함

⚠ 캘리브레이션 후 카메라 간격·각도가 바뀌면 다시 해야 합니다. 명령줄로 같은 계산을 하려면 팀 폴더의 `stereo_calibrate.py` 를 쓰세요.


In [ ]:
#@title C1. 캘리브레이션 사진 폴더 지정 { display-mode: "form" }
CALIB_IMG_DIR = "calib"   #@param {type:"string"}
#@markdown ↑ 체커보드 사진 폴더. `L_00.png`/`R_00.png` … 또는 이름에 `left`/`right` 가 든 파일을 짝지음
BOARD_COLS = 9      #@param {type:"integer"}
BOARD_ROWS = 6      #@param {type:"integer"}
#@markdown ↑ 체커보드 **내부 코너** 개수 (10×7칸 보드 → 9, 6)
SQUARE_MM = 24.2    #@param {type:"number"}
#@markdown ↑ 체커보드 한 칸의 실제 크기(mm)
CALIB_WIDTH = 640   #@param {type:"integer"}
#@markdown ↑ 캘리브레이션 해상도(가로). 측정 사진도 자동으로 이 크기로 맞춤. ESP32-CAM VGA면 640 (0 = 원본)

IMG_EXT = (".png", ".jpg", ".jpeg", ".bmp")

def natural_key(p):
    # IMG_2 < IMG_10 순서로 정렬
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", os.path.basename(p))]

def side_of(p):
    n = os.path.basename(p).lower()
    if n.startswith("l_") or "left" in n: return "L"
    if n.startswith("r_") or "right" in n: return "R"
    return None

imgs = [p for p in glob.glob(os.path.join(CALIB_IMG_DIR, "**", "*"), recursive=True) if p.lower().endswith(IMG_EXT)]
rights = {}
for p in imgs:
    if side_of(p) == "R":
        rights[re.sub(r"(?i)^r_|right", "#", os.path.basename(p), count=1)] = p
calib_pairs = []
for p in sorted(imgs, key=natural_key):
    if side_of(p) == "L":
        key = re.sub(r"(?i)^l_|left", "#", os.path.basename(p), count=1)
        if key in rights:
            calib_pairs.append((p, rights[key]))

print(f"짝지어진 사진: {len(calib_pairs)}쌍  (폴더: {os.path.abspath(CALIB_IMG_DIR)})")
for l, r in calib_pairs[:5]:
    print(f"  {os.path.basename(l)}  ↔  {os.path.basename(r)}")
if len(calib_pairs) > 5:
    print("  ...")
if len(calib_pairs) < 10:
    print("⚠ 10쌍 이상 필요합니다 (권장 20~30쌍)")


In [ ]:
#@title C2. 캘리브레이션 실행
if not calib_pairs:
    raise RuntimeError("캘리브레이션 사진이 없습니다 — C2·C3 를 건너뛰고 '2. 측정할 사진 지정' 셀부터 실행하세요")
MAX_DY_PX = 3.0        #@param {type:"number"}
#@markdown ↑ 불량 쌍 기준: 쌍별 자세로 평행화했을 때 코너 세로 어긋남(px)이 이보다 크면 제외 (두 사진 사이에 보드/카메라가 움직인 쌍 — 한 쌍만 있어도 stereoCalibrate 가 엉뚱한 답으로 수렴함)
CALIB_METHOD = "auto"  #@param ["auto", "stereo", "median"]
#@markdown ↑ auto: stereoCalibrate 와 쌍별 중앙값 자세를 교차검증해 나은 쪽 저장 (두 해의 간격이 5% 넘게 다르면 쌍별 자세). stereo / median 으로 강제 가능

import json
BOARD = (BOARD_COLS, BOARD_ROWS)
objp = np.zeros((BOARD[0] * BOARD[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:BOARD[0], 0:BOARD[1]].T.reshape(-1, 2) * SQUARE_MM
crit = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 100, 1e-5)

def load_calib_img(p):
    img = cv2.imread(p, cv2.IMREAD_COLOR)
    if img is not None and CALIB_WIDTH and img.shape[1] != CALIB_WIDTH:
        s = CALIB_WIDTH / img.shape[1]
        img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    return img

def find_corners(gray):
    # 1차: 정확도가 높은 SB 알고리즘, 실패하면 2차: 기존 알고리즘 + 서브픽셀 보정
    ok, c = cv2.findChessboardCornersSB(gray, BOARD, cv2.CALIB_CB_NORMALIZE_IMAGE | cv2.CALIB_CB_EXHAUSTIVE)
    if not ok:
        ok, c = cv2.findChessboardCorners(gray, BOARD, cv2.CALIB_CB_ADAPTIVE_THRESH | cv2.CALIB_CB_NORMALIZE_IMAGE)
        if ok:
            c = cv2.cornerSubPix(gray, c, (11, 11), (-1, -1), crit)
    if ok:
        c = c.reshape(-1, 1, 2).astype(np.float32)   # 알고리즘마다 배열 모양이 달라서 통일
    return ok, c

def rot_deg(R):
    return float(np.degrees(np.arccos(np.clip((np.trace(R) - 1) / 2, -1, 1))))

def rel_poses(rvL, tvL, rvR, tvR):
    # 쌍마다 (왼쪽 → 오른쪽 카메라) 상대 자세 [(회전벡터, T)]
    out = []
    for i in range(len(rvL)):
        RL, RR = cv2.Rodrigues(rvL[i])[0], cv2.Rodrigues(rvR[i])[0]
        Rr = RR @ RL.T
        out.append((cv2.Rodrigues(Rr)[0].ravel(), (tvR[i] - Rr @ tvL[i]).ravel()))
    return out

def median_pose(rels):
    rv = np.median([r[0] for r in rels], axis=0)
    T = np.median([r[1] for r in rels], axis=0).reshape(3, 1)
    return cv2.Rodrigues(rv)[0], T

def rect_dy(Lp, Rp, K1, D1, K2, D2, size, Rs, T):
    # R, T 로 평행화했을 때 쌍별 코너 세로 어긋남 중앙값(px)과 시차 중앙값(px)
    R1, R2, P1, P2, Q, _, _ = cv2.stereoRectify(K1, D1, K2, D2, size, Rs, T, alpha=0)
    dys, disps = [], []
    for i in range(len(Lp)):
        ra = cv2.undistortPoints(Lp[i], K1, D1, R=R1, P=P1).reshape(-1, 2)
        rb = cv2.undistortPoints(Rp[i], K2, D2, R=R2, P=P2).reshape(-1, 2)
        dys.append(float(np.median(np.abs(ra[:, 1] - rb[:, 1]))))
        disps.append(float(np.median(ra[:, 0] - rb[:, 0])))
    return dys, disps, (R1, R2, P1, P2, Q)

# ---- 코너 ----
names, pts_l, pts_r, previews = [], [], [], []
img_size = None
for pl_path, pr_path in calib_pairs:
    a, b = load_calib_img(pl_path), load_calib_img(pr_path)
    if a is None or b is None or a.shape != b.shape:
        print(f"✖ {os.path.basename(pl_path)}: 읽기 실패 또는 좌우 크기 다름"); continue
    if img_size is None:
        img_size = (a.shape[1], a.shape[0])
    elif (a.shape[1], a.shape[0]) != img_size:
        print(f"✖ {os.path.basename(pl_path)}: 다른 사진과 해상도가 다름"); continue
    ga, gb = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)
    oka, ca = find_corners(ga)
    okb, cb = find_corners(gb)
    if not (oka and okb):
        print(f"✖ {os.path.basename(pl_path)}: 코너 인식 실패 (L:{oka} R:{okb})"); continue
    # 좌우에서 코너 번호 순서가 반대로 잡힌 경우 뒤집기
    if np.dot((ca[-1] - ca[0]).ravel(), (cb[-1] - cb[0]).ravel()) < 0:
        cb = cb[::-1].copy()
    names.append(os.path.basename(pl_path)[:-4]); pts_l.append(ca); pts_r.append(cb)
    if len(previews) < 3:
        va, vb = a.copy(), b.copy()
        cv2.drawChessboardCorners(va, BOARD, ca, True); cv2.drawChessboardCorners(vb, BOARD, cb, True)
        previews.append(np.hstack([va, vb]))

print(f"\n코너 인식 성공: {len(names)} / {len(calib_pairs)}쌍")
if len(names) < 6:
    raise SystemExit("인식된 쌍이 너무 적습니다. BOARD_COLS/ROWS(내부 코너 수)가 맞는지, 체커보드가 다 보이는지 확인하세요.")

# ---- 단일 카메라 + 쌍별 상대 자세 → 좌우 확인 ----
swapped = False
for attempt in range(2):
    n = len(names)
    eL, K1, D1, rvL, tvL = cv2.calibrateCamera([objp] * n, pts_l, img_size, None, None)
    eR, K2, D2, rvR, tvR = cv2.calibrateCamera([objp] * n, pts_r, img_size, None, None)
    rels = rel_poses(rvL, tvL, rvR, tvR)
    Rm, Tm = median_pose(rels)
    if Tm[0, 0] > 0 and attempt == 0:
        print(f"⚠ 왼쪽/오른쪽이 뒤바뀜(간격 벡터 x = {Tm[0, 0]:+.1f} mm) → L 과 R 을 바꿔서 계산합니다. 측정(셀 2)에서도 왼쪽·오른쪽 사진을 바꿔 올리세요.")
        pts_l, pts_r = pts_r, pts_l
        calib_pairs = [(r, l) for l, r in calib_pairs]
        swapped = True
        continue
    break
bases = [float(np.linalg.norm(r[1])) for r in rels]
print(f"단일 카메라 오차   : 왼쪽 {eL:.3f}px / 오른쪽 {eR:.3f}px")
print(f"쌍별 상대 자세     : 간격 {min(bases):.1f}~{max(bases):.1f} mm, 회전 "
      f"{min(np.degrees(np.linalg.norm(r[0])) for r in rels):.2f}~{max(np.degrees(np.linalg.norm(r[0])) for r in rels):.2f}° "
      f"→ 중앙값 간격 {np.linalg.norm(Tm):.1f} mm, 회전 {rot_deg(Rm):.2f}°")

# ---- 불량 쌍 제외 (중앙값 자세로 평행화했을 때 세로 어긋남) ----
dys, disps, _ = rect_dy(pts_l, pts_r, K1, D1, K2, D2, img_size, Rm, Tm)
keep = [i for i in range(n) if dys[i] <= MAX_DY_PX]
dropped = [(names[i], round(dys[i], 2)) for i in range(n) if i not in keep]
for nm, dy in dropped:
    print(f"  제외: {nm} (평행화 세로 어긋남 {dy:.2f}px — 두 사진 사이에 보드/카메라가 움직였을 가능성)")
if len(keep) < 6:
    raise SystemExit(f"일관된 쌍이 부족합니다 ({len(keep)} < 6). 더 촬영하세요.")
Lk, Rk, names_k = [pts_l[i] for i in keep], [pts_r[i] for i in keep], [names[i] for i in keep]
Rm, Tm = median_pose([rels[i] for i in keep])

# ---- stereoCalibrate vs 쌍별 중앙값 자세 교차검증 ----
rms, _, _, _, _, Rs, Ts, _, _ = cv2.stereoCalibrate([objp] * len(keep), Lk, Rk, K1, D1, K2, D2, img_size,
                                                      criteria=crit, flags=cv2.CALIB_FIX_INTRINSIC)
dy_sc, disp_sc, rect_sc = rect_dy(Lk, Rk, K1, D1, K2, D2, img_size, Rs, Ts)
dy_md, disp_md, rect_md = rect_dy(Lk, Rk, K1, D1, K2, D2, img_size, Rm, Tm)
print(f"stereoCalibrate    : RMS {rms:.3f}px, 간격 {np.linalg.norm(Ts):.1f} mm, 회전 {rot_deg(Rs):.2f}°, 평행화 세로 어긋남 중앙값 {np.median(dy_sc):.2f}px")
print(f"쌍별 중앙값 자세   : 간격 {np.linalg.norm(Tm):.1f} mm, 회전 {rot_deg(Rm):.2f}°, 평행화 세로 어긋남 중앙값 {np.median(dy_md):.2f}px")
base_gap = abs(np.linalg.norm(Ts) - np.linalg.norm(Tm)) / np.linalg.norm(Tm)
if CALIB_METHOD == "auto":
    if base_gap > 0.05:
        # 세로 어긋남은 간격(x 성분)에 둔감하다. 두 해의 간격이 5% 넘게 다르면 쌍별 자세(직접 측정)를 믿는다
        print(f"⚠ 두 해의 간격이 {base_gap * 100:.0f}% 다름 → 쌍별 중앙값 자세를 저장 (CALIB_METHOD = stereo 로 강제 가능)")
        use_stereo = False
    else:
        use_stereo = np.median(dy_sc) <= np.median(dy_md)
else:
    use_stereo = CALIB_METHOD == "stereo"
if use_stereo:
    method, Rf, Tf, dyf, dispf, rect = "stereoCalibrate", Rs, Ts, dy_sc, disp_sc, rect_sc
else:
    method, Rf, Tf, dyf, dispf, rect = "median-of-pairs", Rm, Tm, dy_md, disp_md, rect_md
R1, R2, P1, P2, Qc = rect
CALIB_RESULT = "stereo_calib.npz"
np.savez(CALIB_RESULT, K1=K1, D1=D1, K2=K2, D2=D2, R1=R1, R2=R2, P1=P1, P2=P2, Q=Qc, size=np.array(img_size))

# ---- 결과 ----
baseline = float(np.linalg.norm(Tf))
hfov = lambda f: float(np.degrees(2 * np.arctan(img_size[0] / 2 / f)))
med_dy = float(np.median(dyf))
grade = "우수" if med_dy < 0.5 else ("보통" if med_dy < 1.5 else "나쁨 → 재촬영 권장")
# 쌍별 보드 중심 거리: 보드 자세(왼쪽 카메라, 보드 크기 기준) vs 스테레오(평행화 코너 시차) — 줄자 실측과 대조용
dist = []
ctr_obj = objp.mean(axis=0).reshape(3, 1)
for j, i in enumerate(keep):
    Rb = cv2.Rodrigues(rvL[i])[0]
    z_pnp = float((Rb @ ctr_obj + tvL[i].reshape(3, 1))[2, 0])
    ra = cv2.undistortPoints(Lk[j], K1, D1, R=R1, P=P1).reshape(-1, 2)
    rb = cv2.undistortPoints(Rk[j], K2, D2, R=R2, P=P2).reshape(-1, 2)
    d = ra[:, 0] - rb[:, 0]
    z_st = float(np.mean(Qc[2, 3] / (Qc[3, 2] * d + Qc[3, 3])))
    tilt = float(np.degrees(np.arccos(abs(Rb[2, 2]))))
    dist.append(dict(pair=names_k[j], pnp_cm=round(z_pnp / 10, 1), stereo_cm=round(z_st / 10, 1), tilt_deg=round(tilt, 1)))

print(f"\n===== 캘리브레이션 결과 ({method}) =====")
print(f"사용한 사진 쌍     : {len(keep)}" + ("  ※ 좌우 바꿈" if swapped else ""))
print(f"해상도             : {img_size[0]} x {img_size[1]}")
print(f"평행화 세로 어긋남 : 중앙값 {med_dy:.2f}px  ({grade})")
print(f"측정된 카메라 간격 : {baseline:.1f} mm   (1번 설정값 BASELINE_MM = {BASELINE_MM})")
print(f"초점거리 / 화각    : 왼쪽 {K1[0, 0]:.1f}px ≈ {hfov(K1[0, 0]):.1f}° / 오른쪽 {K2[0, 0]:.1f}px ≈ {hfov(K2[0, 0]):.1f}°")
print(f"시차(평행화, 쌍별) : {[round(v, 1) for v in dispf]} px  — 전부 양수여야 정상")
print("쌍별 보드 중심 거리: 보드자세(PnP) / 스테레오  [기울기]  ← 줄자로 잰 거리와 비교해 보세요")
for dd in dist:
    print(f"  {dd['pair']:<6} {dd['pnp_cm']:5.1f} / {dd['stereo_cm']:5.1f} cm  [{dd['tilt_deg']:4.1f}°]")
print(f"저장               : {CALIB_RESULT}\n")
if abs(baseline - BASELINE_MM) > 0.2 * BASELINE_MM:
    print("⚠ 측정된 카메라 간격이 1번 설정값과 20% 이상 다릅니다. SQUARE_MM 값이나 좌우 사진이 바뀌지 않았는지 확인하세요.")
tilts = [dd["tilt_deg"] for dd in dist]
if sum(t >= 15 for t in tilts) < 3:
    print("⚠ 기울인(15° 이상) 보드 쌍이 3개 미만입니다. 평평한 보드만으로는 초점거리가 정해지지 않아 거리가 수 % 틀릴 수 있습니다 → 20~40° 기울인 쌍을 더 찍으세요.")

with open("stereo_calib_report.json", "w", encoding="utf-8") as f:
    json.dump(dict(method=method, swapped_lr=swapped, pairs_used=names_k, dropped=dropped, size=img_size,
                   square_mm=SQUARE_MM, board=BOARD, stereo_rms_px=round(float(rms), 3), rect_dy_median_px=round(med_dy, 3),
                   rect_dy_per_pair=[round(v, 2) for v in dyf], disparity_per_pair=[round(v, 1) for v in dispf],
                   err_left_px=round(float(eL), 3), err_right_px=round(float(eR), 3), baseline_mm=round(baseline, 2),
                   T_mm=[round(float(v), 2) for v in Tf.ravel()], rot_deg=round(rot_deg(Rf), 3),
                   fx_left=round(float(K1[0, 0]), 1), fx_right=round(float(K2[0, 0]), 1),
                   hfov_left_deg=round(hfov(K1[0, 0]), 2), hfov_right_deg=round(hfov(K2[0, 0]), 2),
                   rect_focal_px=round(float(P2[0, 0]), 1), grade=grade, pair_distances=dist), f, ensure_ascii=False, indent=1)

for p in previews:
    plt.figure(figsize=(12, 4)); plt.imshow(cv2.cvtColor(p, cv2.COLOR_BGR2RGB))
    plt.title("코너 인식 예시 (왼쪽 | 오른쪽) — 색깔 선의 순서가 좌우 같아야 함"); plt.axis("off"); plt.show()


In [ ]:
#@title C3. 평행화 확인
c = np.load(CALIB_RESULT)
sz = tuple(int(v) for v in c["size"])
m1 = cv2.initUndistortRectifyMap(c["K1"], c["D1"], c["R1"], c["P1"], sz, cv2.CV_32FC1)
m2 = cv2.initUndistortRectifyMap(c["K2"], c["D2"], c["R2"], c["P2"], sz, cv2.CV_32FC1)
a, b = load_calib_img(calib_pairs[0][0]), load_calib_img(calib_pairs[0][1])
pair = np.hstack([cv2.remap(a, *m1, cv2.INTER_LINEAR), cv2.remap(b, *m2, cv2.INTER_LINEAR)])
plt.figure(figsize=(14, 5)); plt.imshow(cv2.cvtColor(pair, cv2.COLOR_BGR2RGB))
for y in np.linspace(0, sz[1], 14)[1:-1]:
    plt.axhline(y, color="lime", lw=0.7)
plt.title("평행화 결과: 체커보드 코너가 좌우에서 같은 초록 가로줄 위에 있으면 성공"); plt.axis("off"); plt.show()
print("캘리브레이션 파일:", os.path.abspath(CALIB_RESULT), "— stereo.py measure / stereo_pair.py --calib 에서 그대로 사용 가능")


In [ ]:
#@title 2. 측정할 사진 지정
LEFT_PATH = "사용한 파일/cube_L_C2_20261111.jpg"    # 왼쪽 사진 경로 (스마트큐브: C2 = 피사체를 바라볼 때 왼쪽 카메라)
RIGHT_PATH = "사용한 파일/cube_R_C1_20261111.jpg"   # 오른쪽 사진 경로 (스마트큐브: C1)
# 캘리브레이션 파일: C2 셀에서 방금 만든 것 → 없으면 이 폴더의 stereo_calib.npz (2026-11-11, C2·C1 33쌍) → 없으면 근사 측정
CALIB_PATH = CALIB_RESULT if (CALIB_RESULT and os.path.exists(CALIB_RESULT)) else ("stereo_calib.npz" if os.path.exists("stereo_calib.npz") else None)
# 근사 측정으로 강제하려면:  CALIB_PATH = None

for p in (LEFT_PATH, RIGHT_PATH):
    if not os.path.exists(p):
        raise SystemExit(f"파일이 없습니다: {p}")
print(f"왼쪽: {LEFT_PATH}\n오른쪽: {RIGHT_PATH}\n캘리브레이션: {CALIB_PATH or '없음 (근사 측정)'}")


In [ ]:
#@title 3. 사진 불러오기 + 정렬(평행화)
def load(path):
    img = cv2.imread(path, cv2.IMREAD_COLOR)   # 스마트폰 사진의 회전(EXIF)도 반영됨
    if img is None:
        raise ValueError(f"이미지를 읽을 수 없습니다: {path} (jpg/png만 지원)")
    return img

L0, R0 = load(LEFT_PATH), load(RIGHT_PATH)
if L0.shape != R0.shape:
    print(f"⚠ 두 사진 크기가 다릅니다 {L0.shape[:2]} vs {R0.shape[:2]} → 오른쪽을 왼쪽 크기로 맞춤")
    R0 = cv2.resize(R0, (L0.shape[1], L0.shape[0]))

# --- 캘리브레이션 파일 확인 ---
calib = None
if RECTIFY_MODE == "auto" and CALIB_PATH:
    calib = np.load(CALIB_PATH)
    cw, ch = (int(v) for v in calib["size"])
    if (L0.shape[1], L0.shape[0]) != (cw, ch):
        # 가로세로 비율이 같으면 캘리브레이션 해상도로 맞춤 (같은 카메라로 찍은 큰 원본 사진 등)
        if abs(L0.shape[1] / L0.shape[0] - cw / ch) < 0.01:
            print(f"사진 크기 {L0.shape[1]}x{L0.shape[0]} → 캘리브레이션 크기 {cw}x{ch} 로 맞춤")
            L0 = cv2.resize(L0, (cw, ch), interpolation=cv2.INTER_AREA)
            R0 = cv2.resize(R0, (cw, ch), interpolation=cv2.INTER_AREA)
        else:
            print(f"⚠ 사진 비율 {L0.shape[1]}x{L0.shape[0]} ≠ 캘리브레이션 {cw}x{ch} → 캘리브레이션 사용 불가, y-align으로 진행")
            calib = None
mode = "calib" if calib is not None else ("none" if RECTIFY_MODE == "none" else "y-align")

def match_points(a, b, max_pts=4000):
    # SIFT 특징점 매칭 (Lowe ratio test)
    ga, gb = cv2.cvtColor(a, cv2.COLOR_BGR2GRAY), cv2.cvtColor(b, cv2.COLOR_BGR2GRAY)
    sift = cv2.SIFT_create(nfeatures=max_pts)
    ka, da = sift.detectAndCompute(ga, None)
    kb, db = sift.detectAndCompute(gb, None)
    if da is None or db is None or len(ka) < 10 or len(kb) < 10:
        return np.empty((0, 2)), np.empty((0, 2))
    knn = cv2.BFMatcher().knnMatch(da, db, k=2)
    good = [m for m, n in (p for p in knn if len(p) == 2) if m.distance < 0.75 * n.distance]
    pa = np.float32([ka[m.queryIdx].pt for m in good])
    pb = np.float32([kb[m.trainIdx].pt for m in good])
    return pa, pb

if mode == "calib":
    size = (cw, ch)
    m1 = cv2.initUndistortRectifyMap(calib["K1"], calib["D1"], calib["R1"], calib["P1"], size, cv2.CV_32FC1)
    m2 = cv2.initUndistortRectifyMap(calib["K2"], calib["D2"], calib["R2"], calib["P2"], size, cv2.CV_32FC1)
    L = cv2.remap(L0, *m1, cv2.INTER_LINEAR)
    R = cv2.remap(R0, *m2, cv2.INTER_LINEAR)
    Q = calib["Q"]
    print("✔ 캘리브레이션 파일로 왜곡 보정 + 평행화 완료 (정확 측정 모드)")
else:
    # 계산 해상도로 축소
    if RESIZE_WIDTH and L0.shape[1] > RESIZE_WIDTH:
        s = RESIZE_WIDTH / L0.shape[1]
        L = cv2.resize(L0, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
        R = cv2.resize(R0, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
    else:
        L, R = L0.copy(), R0.copy()
    h, w = L.shape[:2]

    if mode == "y-align":
        # 오른쪽 사진의 '세로 위치'만 보정 (y' = y + a + b*x + c*y)
        # 가로(x)는 건드리지 않으므로 '무한히 먼 곳 = 시차 0' 관계가 유지됨
        pl, pr = match_points(L, R)
        if len(pl) < 20:
            print(f"⚠ 매칭점이 {len(pl)}개뿐이라 세로 보정을 건너뜁니다 (무늬가 많은 장면으로 다시 찍어 보세요)")
        else:
            dy = pl[:, 1] - pr[:, 1]
            A = np.c_[np.ones(len(pr)), pr[:, 0], pr[:, 1]]
            keep = np.abs(dy - np.median(dy)) < 3 * (np.median(np.abs(dy - np.median(dy))) + 1)
            for _ in range(5):   # 이상치 제거하며 반복 최소제곱
                coef, *_ = np.linalg.lstsq(A[keep], dy[keep], rcond=None)
                res = dy - A @ coef
                keep = np.abs(res) < max(2.0, 3 * np.median(np.abs(res[keep])))
            xs, ys = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
            # yL = yR + a + b*x + c*yR  →  yR = (yL - a - b*x) / (1 + c)
            map_y = ((ys - coef[0] - coef[1] * xs) / (1 + coef[2])).astype(np.float32)
            R = cv2.remap(R, xs, map_y, cv2.INTER_LINEAR)
            print(f"✔ 세로 정렬 완료: 매칭점 {keep.sum()}/{len(pl)}개 사용, "
                  f"보정 전 세로 어긋남 중앙값 {np.median(np.abs(dy)):.1f}px → 보정 후 {np.median(np.abs(res[keep])):.2f}px")
    else:
        print("정렬 없이 진행 (두 사진이 이미 평행하다고 가정)")

    f_px = (w / 2) / np.tan(np.radians(HFOV_DEG) / 2)
    # Z = f*B/d 를 reprojectImageTo3D 형식의 Q 행렬로 구성
    Q = np.float32([[1, 0, 0, -w / 2],
                    [0, 1, 0, -h / 2],
                    [0, 0, 0, f_px],
                    [0, 0, 1 / BASELINE_MM, 0]])
    print(f"근사 측정 모드: 초점거리 ≈ {f_px:.0f}px (화각 {HFOV_DEG}°), 카메라 간격 {BASELINE_MM}mm")

# 정렬 확인: 같은 물체가 같은 가로줄 위에 있으면 OK
pair = np.hstack([L, R])
fig, ax = plt.subplots(figsize=(14, 5))
ax.imshow(cv2.cvtColor(pair, cv2.COLOR_BGR2RGB))
for y in np.linspace(0, L.shape[0], 12)[1:-1]:
    ax.axhline(y, color="lime", lw=0.7)
ax.set_title("정렬 확인: 같은 물체가 초록 가로줄 기준으로 같은 높이에 있어야 함   (왼쪽 | 오른쪽)")
ax.axis("off"); plt.show()

In [ ]:
#@title 4. 시차(disparity) 계산
gl = cv2.cvtColor(L, cv2.COLOR_BGR2GRAY)
gr = cv2.cvtColor(R, cv2.COLOR_BGR2GRAY)
h, w = gl.shape

if mode == "calib":
    # 정확 측정 모드: 탐색 범위는 거리 범위(MIN_DIST_CM~MAX_DIST_CM)로 계산. 매칭점으로 추정하면 반복 무늬 바닥에서
    # 틀릴 수 있고(2026-11-11 큐브 바닥에서 전부 매칭 실패), 좌우도 캘리브레이션이 정한 것이므로 바꾸지 않음
    f_px, base_mm = float(Q[2, 3]), 1.0 / abs(float(Q[3, 2]))
    max_d = f_px * base_mm / (MIN_DIST_CM * 10)
    scene_min_d = f_px * base_mm / (MAX_DIST_CM * 10)   # 가장 먼 거리의 시차
    min_disp = 0
    num_disp = int(np.ceil((max_d + 16) / 16) * 16)
    print(f"정확 측정 모드: 초점거리 {f_px:.0f}px, 간격 {base_mm:.1f}mm → {MIN_DIST_CM}~{MAX_DIST_CM}cm 는 시차 {max_d:.0f}~{scene_min_d:.1f}px")
else:
    # 근사 모드: 매칭점으로 실제 시차 범위를 측정해서 SGBM 탐색 범위를 자동 설정
    pl, pr = match_points(L, R)
    d_pts = pl[:, 0] - pr[:, 0] if len(pl) else np.array([])
    d_pts = d_pts[np.abs(pl[:, 1] - pr[:, 1]) < 2] if len(pl) else d_pts
    if len(d_pts) >= 10:
        if np.median(d_pts) < 0:
            print("⚠ 시차가 음수입니다 → 왼쪽/오른쪽 사진이 바뀐 것 같아 자동으로 교체합니다")
            L, R, gl, gr = R, L, gr, gl
            d_pts = -d_pts
        lo, hi = np.percentile(d_pts, [1, 99.5])
        min_disp = int(max(0, np.floor(lo) - 8))
        scene_min_d = lo                                # 장면에서 가장 먼 곳의 시차
        # 매칭점이 없는 더 가까운 물체도 잡도록 위쪽은 넉넉하게 (x1.5 + 16px)
        num_disp = int(np.ceil((hi * 1.5 + 16 - min_disp) / 16) * 16)
    else:
        print("⚠ 매칭점이 부족해 기본 탐색 범위를 사용합니다")
        min_disp, num_disp = 0, 128
        scene_min_d = 0
num_disp = int(min(max(num_disp, 16), (w - min_disp) // 16 * 16))
print(f"시차 탐색 범위: {min_disp} ~ {min_disp + num_disp} px")

bs = BLOCK_SIZE
sgbm = cv2.StereoSGBM_create(
    minDisparity=min_disp, numDisparities=num_disp, blockSize=bs,
    P1=8 * 3 * bs ** 2, P2=32 * 3 * bs ** 2, disp12MaxDiff=1,
    uniquenessRatio=10, speckleWindowSize=100, speckleRange=2,
    mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY)
# 왼쪽 가장자리도 계산되도록, 탐색 범위만큼 왼쪽에 검은 여백을 붙여 계산한 뒤 잘라냄
# (여백이 없으면 OpenCV가 왼쪽 min_disp + num_disp 픽셀을 통째로 '측정 불가'로 처리)
PAD = min_disp + num_disp
glp = cv2.copyMakeBorder(gl, 0, 0, PAD, 0, cv2.BORDER_CONSTANT, value=0)
grp = cv2.copyMakeBorder(gr, 0, 0, PAD, 0, cv2.BORDER_CONSTANT, value=0)
raw = sgbm.compute(glp, grp)

# opencv-contrib가 있으면 WLS 필터로 빈 곳을 메우고 경계를 다듬음
if hasattr(cv2, "ximgproc"):
    right_m = cv2.ximgproc.createRightMatcher(sgbm)
    wls = cv2.ximgproc.createDisparityWLSFilter(sgbm)
    wls.setLambda(8000); wls.setSigmaColor(1.2)
    raw = wls.filter(raw, glp, disparity_map_right=right_m.compute(grp, glp))
    print("✔ WLS 필터 적용")
raw = raw[:, PAD:]                                   # 여백 잘라내기
disp = raw.astype(np.float32) / 16.0
valid_disp = disp > max(min_disp, 0.5)
# 짝이 되는 점이 오른쪽 사진 밖에 있으면(물리적으로 오른쪽 카메라에 안 보이는 영역) 제외
valid_disp &= (np.arange(w)[None, :] - disp) >= 0
# 가장 먼 배경보다도 왼쪽에 있는 열은 어떤 물체도 오른쪽 사진에 보일 수 없음 → 제외
valid_disp[:, :int(np.ceil(scene_min_d))] = False
print(f"유효 픽셀 비율: {valid_disp.mean() * 100:.0f}%")
cols = np.where(valid_disp.any(axis=0))[0]
if len(cols):
    print(f"측정 가능한 가로 범위: {cols[0]} ~ {cols[-1]} px (왼쪽 {cols[0]}px는 오른쪽 카메라에 안 보이는 영역)")

In [ ]:
#@title 5. 거리 계산
# 시차 보정: 근사 모드에서만 실제 시차 = 측정 시차 + DISP_OFFSET_PX (정확 측정 모드는 캘리브레이션이 대신하므로 0)
offset = DISP_OFFSET_PX if mode != "calib" else 0.0
print(f"시차 보정값: {offset}px" + ("" if mode != "calib" else " (정확 측정 모드 → 적용 안 함)"))
disp_corr = np.where(valid_disp, disp + offset, 0).astype(np.float32)
pts3d = cv2.reprojectImageTo3D(disp_corr, Q)
Z_cm = pts3d[:, :, 2] / 10.0                      # mm → cm
valid = valid_disp & np.isfinite(Z_cm) & (Z_cm > 0) & (Z_cm < MAX_DIST_CM)
depth_cm = np.where(valid, Z_cm, np.nan)

def distance_at(x, y, roi=7):
    # (x, y) 주변 roi x roi 영역의 유효 거리 중앙값 (cm)
    r = roi // 2
    patch = depth_cm[max(0, y - r):y + r + 1, max(0, x - r):x + r + 1]
    vals = patch[np.isfinite(patch)]
    return float(np.median(vals)) if vals.size >= max(1, patch.size // 4) else None

if not valid.any():
    raise SystemExit("유효한 거리값이 없습니다. 3번 정렬 그림과 설정값(BASELINE_MM, HFOV_DEG)을 확인하세요.")
d_c = distance_at(w // 2, h // 2, roi=21)
print(f"측정된 거리 범위: {np.nanmin(depth_cm):.1f} ~ {np.nanmax(depth_cm):.1f} cm")
print(f"화면 중앙 거리: {d_c:.1f} cm" if d_c else "화면 중앙: 측정 불가")

In [ ]:
#@title 6. 결과 보기
fig, axs = plt.subplots(1, 3, figsize=(18, 5))
axs[0].imshow(cv2.cvtColor(L, cv2.COLOR_BGR2RGB)); axs[0].set_title("왼쪽 (정렬 후)")
dv = np.where(valid_disp, disp, np.nan)
im1 = axs[1].imshow(dv, cmap="jet"); axs[1].set_title("시차 (px) — 빨강 = 가까움")
plt.colorbar(im1, ax=axs[1], fraction=0.046)
im2 = axs[2].imshow(depth_cm, cmap="turbo_r"); axs[2].set_title("거리 (cm) — 빨강 = 가까움")
plt.colorbar(im2, ax=axs[2], fraction=0.046)
for a in axs: a.axis("off")
plt.tight_layout(); plt.show()

# 마우스를 올리면 거리(cm)가 표시되는 인터랙티브 그림
import plotly.graph_objects as go
step = max(1, w // 400)                            # 너무 크면 브라우저가 느려지므로 축소
rgb = cv2.cvtColor(L, cv2.COLOR_BGR2RGB)[::step, ::step]
dsub = depth_cm[::step, ::step]
fig = go.Figure()
fig.add_trace(go.Image(z=rgb, x0=0, y0=0, dx=step, dy=step, hoverinfo="skip"))
fig.add_trace(go.Heatmap(z=dsub, x=np.arange(0, w, step), y=np.arange(0, h, step),
                         colorscale="Turbo", reversescale=True, opacity=0.35,
                         colorbar=dict(title="cm"),
                         hovertemplate="x=%{x}, y=%{y}<br>거리: %{z:.1f} cm<extra></extra>"))
fig.update_layout(title="마우스를 올리면 거리 표시 (색이 없는 곳 = 측정 불가)",
                  width=900, height=int(900 * h / w) + 60, margin=dict(l=0, r=0, t=40, b=0))
fig.update_yaxes(autorange="reversed", scaleanchor="x", showticklabels=False)
fig.update_xaxes(showticklabels=False)
fig.show()

In [ ]:
#@title 7. 원하는 지점 거리 측정 { run: "auto" }
#@markdown 위 그림에서 마우스로 확인한 x, y 좌표를 입력하세요
POINT_X = 320  #@param {type:"integer"}
POINT_Y = 240  #@param {type:"integer"}
ROI = 9        #@param {type:"slider", min:1, max:31, step:2}

x, y = int(np.clip(POINT_X, 0, w - 1)), int(np.clip(POINT_Y, 0, h - 1))
d = distance_at(x, y, ROI)
view = cv2.cvtColor(L, cv2.COLOR_BGR2RGB).copy()
cv2.drawMarker(view, (x, y), (0, 255, 0), cv2.MARKER_CROSS, 30, 2)
plt.figure(figsize=(9, 7)); plt.imshow(view); plt.axis("off")
plt.title(f"({x}, {y}) 거리: {d:.1f} cm" if d else f"({x}, {y}) 측정 불가 — 무늬가 있는 곳을 골라 보세요")
plt.show()

In [ ]:
#@title 8. 결과 저장
np.save(os.path.join(OUT_DIR, "depth_cm.npy"), depth_cm)
vis = cv2.applyColorMap(cv2.normalize(np.nan_to_num(-depth_cm, nan=np.nanmin(-depth_cm)), None, 0, 255,
                                      cv2.NORM_MINMAX).astype(np.uint8), cv2.COLORMAP_TURBO)
cv2.imwrite(os.path.join(OUT_DIR, "depth_color.png"), vis)
print("저장:", os.path.abspath(OUT_DIR), "→ depth_cm.npy, depth_color.png")


## 결과가 이상할 때

| 증상 | 해결 |
|---|---|
| 3번 그림에서 같은 물체 높이가 다름 | 카메라를 같은 높이·방향으로 다시 촬영, 또는 캘리브레이션 파일 사용 |
| 시차 맵이 대부분 비어 있음 | 무늬가 많은 장면, 밝은 곳에서 촬영. `BLOCK_SIZE` 7~9로 키우기. 큐브의 반복 무늬 바닥·거울 벽은 원래 안 잡힘 → 식물·체커보드처럼 질감 있는 대상 기준으로 볼 것 |
| 거리가 일정 비율로 틀림 | 근사 모드: `HFOV_DEG`, `BASELINE_MM` 확인. 정확 측정 모드: 캘리브레이션 쌍에 기울인 보드가 없으면 초점거리가 안 정해짐 → 20~40° 기울인 쌍을 더 넣어 C2 다시 (`SQUARE_MM` 도 확인) |
| 가까운 물체가 측정 안 됨 | 너무 가까움 (간격 33mm·640px 기준 약 20cm 이상). 정확 측정 모드는 `MIN_DIST_CM` 을 낮추기 |
| 좌우가 바뀜 | 근사 모드: 4번 셀이 자동 감지해 교체. 정확 측정 모드: 자동 교체 안 함 → 거리값이 거의 없거나 전부 멀게 나오면 왼쪽·오른쪽 사진을 바꿔 올리기 (스마트큐브: 왼쪽 = C2, 오른쪽 = C1) |
| 캘리브레이션 간격이 실측과 다름 | `SQUARE_MM` 오류(프린터 배율), 또는 두 사진 사이에 보드가 움직인 쌍 → C2 출력의 "제외" 줄 확인 |
